|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 1:</h2>|<h1>The Naive Loop<h1>|
|<h2>Section:</h2>|<h1>The roofline<h1>|
|<h2>Lecture:</h2>|<h1><b>Measure your own card: bandwidth, compute, and the ridge<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import time
import numpy as np
import torch
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib

# Do not use the numbers on the box

The last two notebooks used placeholder figures. Now measure your own card.
The spec sheet and the machine disagree. The machine is correct.

In [2]:
properties = torch.cuda.get_device_properties(0)
print(f'{properties.name}')
print(f'  VRAM        {properties.total_memory/1e9:.1f} GB')
print(f'  SMs         {properties.multi_processor_count}')
print(f'  L2 cache    {properties.L2_cache_size/1e6:.0f} MB')
print(f'  compute cap {properties.major}.{properties.minor}')

NVIDIA GeForce RTX 4080 Laptop GPU
  VRAM        12.5 GB
  SMs         58
  L2 cache    50 MB
  compute cap 8.9


### Bandwidth: copy a large block and time it

The copy moves 256 MB in and 256 MB out. No cache holds that much. This is the
only way to measure memory instead of cache.

In [3]:
num_values = 256*1024*1024 // 2                       # 256 MB of bf16
source = torch.empty(num_values, dtype=torch.bfloat16, device='cuda')
target = torch.empty_like(source)

copy_ms = cudalib.bench_ms(lambda: target.copy_(source), best_of=3)
bandwidth = (2 * source.numel() * 2) / (copy_ms*1e-3)          # read + write
print(f'{copy_ms:.3f} ms to move {4*num_values/1e6:.0f} MB  ->  {bandwidth/1e9:.0f} GB/s')

del source, target
torch.cuda.empty_cache()

1.591 ms to move 537 MB  ->  337 GB/s


### Compute: measure it twice, and here is why

A laptop GPU first runs at a high clock. It then becomes hot and reduces the
clock. So one number does not exist.

A server holds a sustained load. The second number is therefore the honest
one.

In [4]:
matrix = torch.randn(4096, 4096, dtype=torch.bfloat16, device='cuda')
MATMUL_FLOP = 2 * 4096**3

burst = MATMUL_FLOP / (cudalib.bench_ms(lambda: matrix@matrix, iters=30, warmup=20)*1e-3)

end = time.perf_counter() + 3.0                 # heat it up on purpose
while time.perf_counter() < end: matrix@matrix
torch.cuda.synchronize()

sustained = MATMUL_FLOP / (cudalib.bench_ms(lambda: matrix@matrix, iters=100, warmup=0)*1e-3)

print(f'burst     {burst/1e12:5.0f} TFLOP/s')
print(f'sustained {sustained/1e12:5.0f} TFLOP/s   ({100*sustained/burst:.0f}% of burst)')
print('\nA benchmark you run cold will lie to you, in your favour.')

burst        48 TFLOP/s
sustained    35 TFLOP/s   (71% of burst)

A benchmark you run cold will lie to you, in your favour.


# Your ridge point

In [5]:
ridge = sustained / bandwidth
print(f'  {sustained/1e12:6.1f} TFLOP/s')
print(f'  {bandwidth/1e9:6.0f} GB/s')
print(f'  ' + '-'*22)
print(f'  {ridge:6.0f} FLOP per byte\n')
print(f'At batch 1 your workload sits at 1, against {ridge:.0f}.')
print(f'That is {100/ridge:.1f}% of the arithmetic this card can do.')

    34.6 TFLOP/s
     337 GB/s
  ----------------------
     103 FLOP per byte

At batch 1 your workload sits at 1, against 103.
That is 1.0% of the arithmetic this card can do.


### What the ridge costs, in tokens per second

At batch 1 a token costs one read of the weights. Your code cannot change
that. These numbers are ceilings, not estimates.

In [6]:
print(f"{'model':<12} {'GB/step':>8} {'tok/s':>8} {'ms/token':>9}")
for name, weight_gb in [('0.6B bf16',1.2), ('1B',2.0), ('7B',14.0), ('8B fp8',8.0)]:
  tokens_per_second = bandwidth/1e9 / weight_gb
  print(f'{name:<12} {weight_gb:>8.1f} {tokens_per_second:>8.0f} {1000/tokens_per_second:>9.2f}')

model         GB/step    tok/s  ms/token
0.6B bf16         1.2      281      3.56
1B                2.0      169      5.93
7B               14.0       24     41.49
8B fp8            8.0       42     23.71


To pass these ceilings you must raise the batch. Continuous batching exists
for this reason. The next notebook shows the ceiling.